In [0]:
# ════════════════════════════════════════════
# Workshop Configuration — no edits needed
# ════════════════════════════════════════════
import re

# Read the current user directly from the Databricks workspace context
_raw_user   = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
USERNAME    = re.sub(r'[^a-zA-Z0-9]', '_', _raw_user.split("@")[0])

CATALOG     = "workshop"         # shared catalog, pre-configured by facilitator
SCHEMA      = USERNAME           # your personal schema inside the catalog

# MLflow experiment path — lives in your personal workspace folder
EXPERIMENT_PATH = f"/Users/{_raw_user}/gdm_yield_{USERNAME}"

print(f"📍 Logged in as  : {_raw_user}")
print(f"📍 USERNAME       : {USERNAME}")
print(f"📍 Your namespace : {CATALOG}.{SCHEMA}")
print(f"📍 Your table     : {CATALOG}.{SCHEMA}.yield_trials")
print(f"📍 Your model     : gdm-yield-model-{USERNAME}")
print(f"📍 MLflow path    : {EXPERIMENT_PATH}")

In [0]:
# Create your personal schema (run once)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"✅ Schema ready: {CATALOG}.{SCHEMA}")

In [0]:
# Install Kaggle CLI
!pip install kaggle --quiet

# Download the dataset from Kaggle
# Note: This requires Kaggle API credentials (~/.kaggle/kaggle.json)
# If you don't have them, download manually from:
# https://www.kaggle.com/datasets/patelris/crop-yield-prediction-dataset
!kaggle datasets download -d patelris/crop-yield-prediction-dataset

print("✅ Dataset downloaded successfully")

In [0]:
import zipfile
import os

# Extract the ZIP file
zip_path = "crop-yield-prediction-dataset.zip"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(".")
    print(f"✅ Extracted: {zip_ref.namelist()}")

# Load the CSV into a Spark DataFrame using absolute path
csv_path = "/Workspace/Users/valentina.mesa4418@unaula.edu.co/Big_DATA/yield_df.csv"
yield_df = spark.read.csv(csv_path, header=True, inferSchema=True)

print(f"\n✅ DataFrame loaded with {yield_df.count()} rows")

In [0]:
# 1. Show schema (column names and types)
print("📋 SCHEMA:")
yield_df.printSchema()

# 2. Show total row count
total_rows = yield_df.count()
print(f"\n📊 TOTAL ROWS: {total_rows:,}")

# 3. Show 5 sample rows
print("\n📄 SAMPLE ROWS (5):")
display(yield_df.limit(5))

In [0]:
# 4. Show unique values in "Item" column (crop types)
print("🌾 UNIQUE CROP TYPES (Item column):")
unique_items = yield_df.select("Item").distinct().orderBy("Item")
print(f"\nTotal unique crops: {unique_items.count()}")
display(unique_items)

In [0]:
# 5. Show unique values in "Area" that include "Argentina" or "Brazil"
from pyspark.sql.functions import col

print("🌎 AREAS CONTAINING 'Argentina' OR 'Brazil':")
areas_filter = yield_df.filter(
    col("Area").contains("Argentina") | col("Area").contains("Brazil")
).select("Area").distinct().orderBy("Area")

print(f"\nTotal matching areas: {areas_filter.count()}")
display(areas_filter)

In [0]:
# Save the DataFrame as a Delta table
table_name = f"{CATALOG}.{SCHEMA}.yield_trials"
yield_df.write.format("delta").mode("overwrite").saveAsTable(table_name)

print(f"✅ Table saved: {table_name}")

# Sanity check 1: Count total rows
row_count = spark.sql(f"SELECT COUNT(*) as count FROM {table_name}").collect()[0]["count"]
print(f"\n📊 Total rows in table: {row_count:,}")

# Sanity check 2: Show first 5 rows
print(f"\n📄 First 5 rows from {table_name}:")
display(spark.sql(f"SELECT * FROM {table_name} LIMIT 5"))

In [0]:
# Load the table into a pandas DataFrame
table_name = f"{CATALOG}.{SCHEMA}.yield_trials"
df = spark.table(table_name).toPandas()

print(f"✅ Loaded {len(df):,} rows from {table_name}")
print(f"\n📋 Columns: {df.columns.tolist()}")
print(f"\n📊 Shape: {df.shape}")
df.head()

In [0]:
# Define target and feature columns
target_col = "hg/ha_yield"
feature_cols = [
    "Item", 
    "Area", 
    "average_rain_fall_mm_per_year", 
    "pesticides_tonnes", 
    "avg_temp", 
    "Year"
]

print(f"🎯 Target: {target_col}")
print(f"\n🔧 Features ({len(feature_cols)}):")
for col in feature_cols:
    print(f"   • {col}")

# Prepare X and y
X = df[feature_cols]
y = df[target_col]

print(f"\n✅ X shape: {X.shape}")
print(f"✅ y shape: {y.shape}")

In [0]:
%pip install xgboost --quiet

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBRegressor

# Define categorical and numeric columns
categorical_cols = ["Item", "Area"]
numeric_cols = ["average_rain_fall_mm_per_year", "pesticides_tonnes", "avg_temp", "Year"]

# Build preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OrdinalEncoder(
            handle_unknown="use_encoded_value", 
            unknown_value=-1
        ), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

# Build full pipeline: preprocessing + model
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42
    ))
])

print("✅ Pipeline built:")
print(pipeline)

In [0]:
from sklearn.model_selection import train_test_split

# Split data 80/20 (keep raw strings in X_train/X_test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42
)

print("📊 TRAIN/TEST SPLIT SHAPES:")
print(f"\n   X_train: {X_train.shape}")
print(f"   X_test:  {X_test.shape}")
print(f"   y_train: {y_train.shape}")
print(f"   y_test:  {y_test.shape}")

print(f"\n✅ Split complete: {len(X_train):,} train / {len(X_test):,} test")
print(f"   Train ratio: {len(X_train)/len(X)*100:.1f}%")
print(f"   Test ratio:  {len(X_test)/len(X)*100:.1f}%")

In [0]:
import mlflow
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Set MLflow experiment to user's workspace folder
mlflow.set_experiment(EXPERIMENT_PATH)
print(f"📍 MLflow Experiment: {EXPERIMENT_PATH}")

# Start MLflow run
with mlflow.start_run() as run:
    print(f"\n🚀 Training pipeline...")
    
    # Fit the pipeline on training data
    pipeline.fit(X_train, y_train)
    print("✅ Pipeline trained successfully")
    
    # Predict on test set
    y_pred = pipeline.predict(X_test)
    
    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    print(f"\n📊 TEST SET METRICS:")
    print(f"   RMSE: {rmse:,.2f}")
    print(f"   R²:   {r2:.4f}")
    
    # Log hyperparameters
    mlflow.log_params({
        "n_estimators": 100,
        "max_depth": 5,
        "learning_rate": 0.1
    })
    
    # Log metrics
    mlflow.log_metrics({
        "rmse": rmse,
        "r2": r2
    })
    
    # Infer model signature (required for Unity Catalog)
    from mlflow.models import infer_signature
    signature = infer_signature(X_train, y_pred)
    
    # Log the full pipeline (preprocessing + model)
    model_name = f"{CATALOG}.{SCHEMA}.gdm-yield-model-{USERNAME}"
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="gdm_yield_pipeline",
        registered_model_name=model_name,
        signature=signature
    )
    
    print(f"\n✅ Model logged to MLflow")
    print(f"   Registered model: {model_name}")
    print(f"   Run ID: {run.info.run_id}")

In [0]:
import plotly.graph_objects as go

# Extract feature importances from the trained XGBoost model
feature_importances = pipeline.named_steps["regressor"].feature_importances_

# Feature names in ColumnTransformer order: categorical first, then numeric
feature_names = [
    "Crop Type",
    "Country",
    "Rainfall (mm/yr)",
    "Pesticides (t)",
    "Avg Temp (°C)",
    "Year"
]

# Create DataFrame for sorting
import pandas as pd
fi_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": feature_importances
}).sort_values("Importance", ascending=True)  # ascending for horizontal bar

# Create horizontal bar chart
fig = go.Figure([
    go.Bar(
        x=fi_df["Importance"],
        y=fi_df["Feature"],
        orientation='h',
        marker=dict(color='steelblue')
    )
])

fig.update_layout(
    title="Feature Importances (XGBoost)",
    xaxis_title="Importance",
    yaxis_title="Feature",
    height=400,
    showlegend=False
)

fig.show()

In [0]:
import plotly.graph_objects as go

# Create scatter plot: predicted vs actual
fig = go.Figure()

# Add scatter points
fig.add_trace(go.Scatter(
    x=y_test,
    y=y_pred,
    mode='markers',
    marker=dict(
        size=4,
        color='steelblue',
        opacity=0.5
    ),
    name='Predictions'
))

# Add diagonal reference line (perfect prediction)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())

fig.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    line=dict(
        color='red',
        dash='dash',
        width=2
    ),
    name='Perfect Prediction'
))

fig.update_layout(
    title=f"Predicted vs Actual Yield (Test Set)<br><sub>RMSE: {rmse:,.2f} | R²: {r2:.4f}</sub>",
    xaxis_title="Actual Yield (hg/ha)",
    yaxis_title="Predicted Yield (hg/ha)",
    height=500,
    width=600,
    showlegend=True
)

fig.show()